In [25]:
from tqdm import tqdm
# Traditional for loop
squared = []
for i in tqdm(range(10**8)):
    squared.append(i**2)

100%|███████████████████████████████████████████████████████████████| 100000000/100000000 [01:31<00:00, 1090032.31it/s]


In [26]:
# Faster list comprehension
squared = [i**2 for i in tqdm(range(10**8))]

100%|███████████████████████████████████████████████████████████████| 100000000/100000000 [01:16<00:00, 1313627.58it/s]


In [27]:
### Pandas vs Polars

In [28]:
import pandas as pd
import polars as pl
import numpy as np
import time
import os

file_name = 'large_data.csv'

# Ensure the large_data.csv file exists.
# This part of the code is added to generate the file if it doesn't exist,
# so the rest of the script can run without errors.
if not os.path.exists(file_name):
    print(f"'{file_name}' not found. Generating a new one...")
    num_rows = 1_000_000  # 1 million rows
    num_categories = 100
    num_cities = 50

    # Generate data
    data = {
        'value': np.random.randint(1, 1000, num_rows),
        'category': [f'cat_{i}' for i in np.random.randint(0, num_categories, num_rows)],
        'city': [f'city_{i}' for i in np.random.randint(0, num_cities, num_rows)]
    }

    df_large = pd.DataFrame(data)
    df_large.to_csv(file_name, index=False)
    print(f"'{file_name}' created successfully with {num_rows} rows.")
else:
    print(f"'{file_name}' already exists. Skipping file generation.")


print("\n--- Pandas Execution ---")
start_time_pandas = time.time()

# 1. Read data
df_pd = pd.read_csv(file_name)
print(df_pd.shape)
print(df_pd.head())


# 2. Filter data (e.g., value > 500 AND category starts with 'cat_5' AND city is 'city_10')
df_pd_filtered = df_pd[
    (df_pd["value"] > 500) &
    (df_pd["category"].str.startswith("cat_5")) &
    (df_pd["city"] == "city_10")
]

# 3. Group by and aggregate (mean of 'value' by 'category' and 'city')
df_pd_grouped = df_pd_filtered.groupby(["category", "city"])["value"].mean().reset_index()

# 4. Add a new derived column (e.g., scaled_value)
df_pd_grouped["scaled_value"] = df_pd_grouped["value"] * 1.5

end_time_pandas = time.time()
pandas_execution_time = end_time_pandas - start_time_pandas
print(f"Pandas total execution time: {pandas_execution_time:.4f} seconds")


# --- Polars Execution (starting from Pandas DataFrame) ---
print("\n--- Polars Execution (with pandas to polars conversion) ---")
start_time_polars = time.time()

# 1. Convert Pandas DataFrame to Polars DataFrame
# This is the step you wanted to include.
# Note: df_pl is an eager Polars DataFrame immediately after this conversion.
df_pl = pl.from_pandas(df_pd)
print(f"Polars initial shape (from Pandas): {df_pl.shape}")
print(df_pl.head()) # Commented out to reduce print time if not needed


# 2. Convert to LazyFrame to leverage Polars' optimization
# This is crucial for performance after the initial eager conversion.
df_pl_lazy = df_pl.lazy()

# 3. Filter data (operations on LazyFrame build the query plan)
df_pl_filtered_lazy = df_pl_lazy.filter(
    (pl.col("value") > 500) &
    (pl.col("category").str.starts_with("cat_5")) &
    (pl.col("city") == "city_10")
)

# 4. Group by and aggregate
df_pl_grouped_lazy = df_pl_filtered_lazy.group_by(["category", "city"]).agg(
    pl.col("value").mean().alias("value")
)

# 5. Add a new derived column and trigger computation with .collect()
df_pl_final = df_pl_grouped_lazy.with_columns(
    (pl.col("value") * 1.5).alias("scaled_value")
).collect() # .collect() triggers the actual execution of the optimized query plan

# print(f"Polars final shape: {df_pl_final.shape}")
# print(df_pl_final.head()) # Commented out to reduce print time if not needed

end_time_polars = time.time()
polars_execution_time = end_time_polars - start_time_polars
print(f"Polars total execution time: {polars_execution_time:.4f} seconds")

# --- Comparison ---
print("\n--- Comparison ---")
print(f"Pandas took: {pandas_execution_time:.4f} seconds")
print(f"Polars took: {polars_execution_time:.4f} seconds")

if pandas_execution_time > polars_execution_time:
    speed_up = pandas_execution_time / polars_execution_time
    print(f"Polars was approximately {speed_up:.2f} times faster than Pandas for this task.")
else:
    print("In this specific case, Pandas was faster or similar (less common for large datasets and complex operations).")


--- Pandas Execution ---
(5000000, 5)
   id       value category      city            timestamp
0   0  119.256913   cat_54  city_401  2020-12-22 21:56:45
1   1  298.987514   cat_57  city_397  2020-10-14 10:16:11
2   2   51.337477   cat_23  city_272  2020-04-22 04:46:09
3   3    5.448552   cat_37   city_35  2020-12-21 06:25:55
4   4  694.699538   cat_66  city_115  2020-10-06 21:41:10
Pandas total execution time: 18.3831 seconds

--- Polars Execution (with pandas to polars conversion) ---
Polars initial shape (from Pandas): (5000000, 5)
shape: (5, 5)
┌─────┬────────────┬──────────┬──────────┬─────────────────────┐
│ id  ┆ value      ┆ category ┆ city     ┆ timestamp           │
│ --- ┆ ---        ┆ ---      ┆ ---      ┆ ---                 │
│ i64 ┆ f64        ┆ str      ┆ str      ┆ str                 │
╞═════╪════════════╪══════════╪══════════╪═════════════════════╡
│ 0   ┆ 119.256913 ┆ cat_54   ┆ city_401 ┆ 2020-12-22 21:56:45 │
│ 1   ┆ 298.987514 ┆ cat_57   ┆ city_397 ┆ 2020-10-14 